<a href="https://colab.research.google.com/github/pxs1990/NLP_LLM/blob/main/training_RAG_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RAG pipeline with AWS Bedrock LLM
# Step 1: Imports
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
#Step 2: Initialize AWS Bedrock Models
# Bedrock LLM (Amazon Nova)
llm_nova = ChatBedrockConverse(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1",
    temperature=0.5,
    max_tokens=200
)
# Bedrock Embeddings (Titan)
embeddings_titan = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v1"
)

#Step 3: Create Source Documents
texts = [
    "Employees are entitled to 20 days of paid leave per year.",
    "Employees may work from home up to 3 days a week with manager approval.",
    "All employees must complete security training every 6 months."
]
documents = [Document(page_content=text) for text in texts]

#Step 4: Create Vector Store (FAISS)
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings_titan
)

#Step 5: Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

#Step 6: Retrieve Relevant Documents
query = "How many days of paid leave do employees get?"
retrieved_docs = retriever.invoke(query)

#Step 7: Build RAG Prompt (Context + Question)
context = "\n".join(doc.page_content for doc in retrieved_docs)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the provided context."),
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])

#Step 8: Generate Final Answer
response = llm_nova.invoke(
    prompt.format_messages(
        context=context,
        question=query
    )
)
print("RAG Answer:\n", response.content)


In [ ]:
# RAG pipeline with hybrid search and re-reanking
# ============ STEP 1: IMPORTS ============
import os
from typing import List, Dict, Tuple
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import CrossEncoder  # For re-ranking
import torch
import numpy as np

# ============ STEP 2: INITIALIZE MODELS (FREE/LOCAL) ============
# Use lightweight models for Colab/Laptop

# 1. Embedding Model (MiniLM - lightweight)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},  # Use 'cuda' if available
    encode_kwargs={'normalize_embeddings': True}
)

# 2. LLM (Llama 3.2 3B - lightweight for Colab)
llm = ChatHuggingFace.from_model_id(
    model_id="meta-llama/Llama-3.2-3B-Instruct",
    task="text-generation",
    device_map="auto",
    model_kwargs={
        "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
        "load_in_4bit": True,  # Quantization for memory efficiency
        "temperature": 0.1,
        "max_new_tokens": 512,
        "do_sample": True
    }
)

# 3. Re-ranker Model (Lightweight cross-encoder)
class LightweightReranker:
    def __init__(self, model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"):
        from sentence_transformers import CrossEncoder
        self.model = CrossEncoder(model_name, max_length=512)

    def rerank(self, query: str, documents: List[Document], top_k: int = 5) -> List[Tuple[Document, float]]:
        """Re-rank documents based on relevance to query"""
        pairs = [(query, doc.page_content) for doc in documents]
        scores = self.model.predict(pairs)

        # Combine documents with scores
        scored_docs = list(zip(documents, scores))
        # Sort by score (descending)
        scored_docs.sort(key=lambda x: x[1], reverse=True)

        return scored_docs[:top_k]

reranker = LightweightReranker()

# ============ STEP 3: CREATE SOURCE DOCUMENTS (10+ TEXTS) ============
texts = [
    # Original 3
    "Employees are entitled to 20 days of paid leave per year.",
    "Employees may work from home up to 3 days a week with manager approval.",
    "All employees must complete security training every 6 months.",

    # Additional 10 texts
    "Health insurance coverage includes medical, dental, and vision plans for all full-time employees.",
    "Annual performance reviews are conducted in December, with bonuses distributed in January.",
    "The company offers a 401(k) retirement plan with 4% employer matching contribution.",
    "New employees receive a $2,000 relocation allowance if moving more than 50 miles.",
    "All technical staff must obtain at least one professional certification within their first year.",
    "The standard work schedule is Monday to Friday, 9 AM to 5 PM, with flexible start times.",
    "Employees can access up to $5,000 per year in tuition reimbursement for job-related courses.",
    "Maternity leave is 16 weeks paid, paternity leave is 8 weeks paid.",
    "The office dress code is business casual, except for client meetings which require formal attire.",
    "All software engineers receive a $3,000 annual budget for home office equipment.",
    "Company-wide team building retreats occur twice per year, in spring and fall.",
    "Employees receive 15 sick days per year, which can also be used for family medical needs.",
    "Promotion cycles occur every 6 months, with eligibility based on performance metrics.",
]

documents = [Document(page_content=text, metadata={"source": f"doc_{i}"})
             for i, text in enumerate(texts)]

# ============ STEP 4: CREATE HYBRID RETRIEVAL SYSTEM ============
def create_hybrid_retriever(docs: List[Document], k: int = 10) -> EnsembleRetriever:
    """Create a hybrid retriever combining dense and sparse search"""

    # 1. Dense retriever (semantic search with FAISS)
    vectorstore = FAISS.from_documents(
        documents=docs,
        embedding=embeddings
    )
    dense_retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )

    # 2. Sparse retriever (keyword search with BM25)
    bm25_retriever = BM25Retriever.from_documents(
        documents=docs,
        k=k
    )

    # 3. Combine both retrievers (hybrid search)
    ensemble_retriever = EnsembleRetriever(
        retrievers=[dense_retriever, bm25_retriever],
        weights=[0.6, 0.4]  # Weighted combination
    )

    return ensemble_retriever, vectorstore

# Create hybrid retriever
hybrid_retriever, faiss_index = create_hybrid_retriever(documents, k=10)

# ============ STEP 5: BUILD COMPLETE RAG PIPELINE ============
def rag_pipeline_with_reranking(
    query: str,
    retriever,
    llm,
    reranker=None,
    use_reranking: bool = True,
    top_k_retrieve: int = 10,
    top_k_final: int = 5
) -> Dict:
    """
    Complete RAG pipeline with optional re-ranking
    """

    # Step 1: Initial retrieval (hybrid search)
    print(f"🔍 Retrieving documents for: '{query}'")
    initial_docs = retriever.invoke(query)

    # Step 2: Re-ranking (CRITICAL for industry)
    if use_reranking and reranker:
        print(f"📊 Re-ranking {len(initial_docs)} documents...")
        reranked_docs_with_scores = reranker.rerank(query, initial_docs, top_k=top_k_final)
        final_docs = [doc for doc, score in reranked_docs_with_scores]
        scores = [score for doc, score in reranked_docs_with_scores]
    else:
        final_docs = initial_docs[:top_k_final]
        scores = [1.0] * len(final_docs)  # Placeholder scores

    # Step 3: Build context
    context = "\n---\n".join([
        f"[Document {i+1}, Score: {scores[i]:.3f}]: {doc.page_content}"
        for i, doc in enumerate(final_docs)
    ])

    # Step 4: Create RAG prompt
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a precise HR assistant. Answer STRICTLY based on the provided context.
         If the answer isn't in the context, say "I cannot find that information."
         Provide concise, accurate answers."""),
        ("human", f"""Context from company documents:
{context}

Question: {query}

Answer based only on the context above:""")
    ])

    # Step 5: Generate answer
    print("🤖 Generating answer...")
    response = llm.invoke(prompt.format_messages())

    # Step 6: Return comprehensive result
    return {
        "answer": response.content,
        "sources": [
            {
                "content": doc.page_content,
                "score": float(scores[i]),
                "metadata": doc.metadata
            }
            for i, doc in enumerate(final_docs)
        ],
        "retrieval_method": "Hybrid (Dense + Sparse) with Re-ranking" if use_reranking else "Hybrid only",
        "total_docs_retrieved": len(initial_docs),
        "docs_after_reranking": len(final_docs)
    }

# ============ STEP 6: TEST QUERIES ============
if __name__ == "__main__":
    print("🚀 Initializing RAG Pipeline...")
    print(f"Embedding model: MiniLM-L6-v2")
    print(f"LLM: Llama-3.2-3B-Instruct")
    print(f"Using hybrid search: ✓")
    print(f"Using re-ranking: ✓\n")

    # Test queries
    test_queries = [
        "How many days of paid leave do employees get?",
        "What is the work from home policy?",
        "Tell me about health insurance benefits",
        "How much tuition reimbursement is available?",
        "What is the maternity leave policy?",
        "When are performance reviews conducted?",
    ]

    for i, query in enumerate(test_queries):
        print(f"\n{'='*60}")
        print(f"QUERY {i+1}: {query}")
        print(f"{'='*60}")

        result = rag_pipeline_with_reranking(
            query=query,
            retriever=hybrid_retriever,
            llm=llm,
            reranker=reranker,
            use_reranking=True,  # Toggle this to see difference
            top_k_retrieve=10,
            top_k_final=3
        )

        print(f"\n✅ ANSWER: {result['answer']}")
        print(f"\n📚 TOP SOURCES (Ranked):")
        for j, source in enumerate(result['sources']):
            print(f"  {j+1}. [Score: {source['score']:.3f}] {source['content'][:100]}...")

        print(f"\n📊 Stats: Retrieved {result['total_docs_retrieved']} docs → {result['docs_after_reranking']} after re-ranking")

# ============ STEP 7: COMPARISON UTILITY ============
def compare_without_reranking(query: str):
    """Compare results with and without re-ranking"""
    print(f"\n🔬 COMPARISON FOR: '{query}'")
    print(f"{'-'*40}")

    # With re-ranking
    print("WITH RE-RANKING:")
    with_result = rag_pipeline_with_reranking(
        query, hybrid_retriever, llm, reranker, use_reranking=True, top_k_final=3
    )
    print(f"Answer: {with_result['answer'][:150]}...")
    print(f"Top source: {with_result['sources'][0]['content'][:100]}...")

    # Without re-ranking
    print(f"\nWITHOUT RE-RANKING:")
    without_result = rag_pipeline_with_reranking(
        query, hybrid_retriever, llm, reranker=None, use_reranking=False, top_k_final=3
    )
    print(f"Answer: {without_result['answer'][:150]}...")
    print(f"Top source: {without_result['sources'][0]['content'][:100]}...")

# Run comparison
print("\n\n📈 RE-RANKING COMPARISON DEMO")
compare_without_reranking("What benefits are available for new employees?")